# SOXL 换手率零值来源纠正

本卡的首次来源门只检验了有限、非负数。现只读已备份原响应、原生 Qlib 来源和 Stage1 日历，检查成交额为正时零换手率的含义；不读取未来结果，不重跑来源请求或账户。

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd(); card=root/"research/stage1_futu_soxl_turnover_source_20261023"
source=json.loads((card/"summary.json").read_text())
receipt=json.loads((root/"data/raw/external/stage1_futu_soxl_turnover_20261023/request_receipt.json").read_text())
raw_path=root/receipt["data_file"]
assert hashlib.sha256(raw_path.read_bytes()).hexdigest()==receipt["data_sha256"]
assert json.loads((card/"nas_scoped_restore.json").read_text())["status"]=="NAS_scoped_restore_and_native_Qlib_readback_passed"
raw=pd.read_csv(raw_path,parse_dates=["time_key"])
zero=raw.turnover_rate.eq(0)
assert zero.sum()==444 and raw.loc[zero,"volume"].gt(0).all() and raw.loc[zero,"turnover"].gt(0).all()
qlib.init(provider_uri=str(root/"data/qlib/stage1_futu_soxl_turnover_20261023"),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=source["experiment_id"],recorder_id=source["recorder_id"])
 native=rec.load_object("analysis/native_turnover_rate")
assert len(native)==len(raw.loc[raw.time_key.ge("2011-01-03")])
q=D.features(["SOXL"],["$turnover_rate"],start_time="2010-03-31",end_time="2026-10-09",freq="day")["$turnover_rate"].droplevel("instrument")
assert q.reindex(native.index).index.equals(native.index) and (q.reindex(native.index).values==native.values).all()
qlib.init(provider_uri=str(root/"data/qlib/minute_information_pit_allocator_20261012/daily_prices"),region="us",expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2022-12-30",freq="day"))
train=native.reindex(calendar)
valid=train.gt(0)
assert len(train)==3020 and valid.sum()==2766
correction={"status":"needs_source_history_turnover_rate_zero_encoded_2011",
 "overrides":"result.json and summary.json conditional_source_coverage_pass_first_public_unknown for usable turnover_rate",
 "source_recorder_id":source["recorder_id"],"source_raw_sha256":receipt["data_sha256"],
 "source_NAS_scoped_restore_verified":True,"all_source_rows":len(raw),
 "zero_rate_rows":int(zero.sum()),"zero_rate_first":str(raw.loc[zero,"time_key"].min().date()),
 "zero_rate_last":str(raw.loc[zero,"time_key"].max().date()),
 "zero_rate_by_year":{str(k):int(v) for k,v in raw.loc[zero].groupby(raw.loc[zero,"time_key"].dt.year).size().items()},
 "zero_rate_rows_with_positive_volume_and_turnover":int((zero&raw.volume.gt(0)&raw.turnover.gt(0)).sum()),
 "Stage1_train_rows":len(train),"Stage1_train_nonzero_known":int(valid.sum()),
 "Stage1_train_required_min":3000,"Stage1_usable_coverage_gate_pass":False,
 "Stage1_train_other_missing_dates":[str(x.date()) for x in train.index[train.isna()]],
 "first_public_historical_qualified":False,"future_outcome_reads":0,"new_fits":0,"new_accounts":0,
 "decision":"close_turnover_rate_field_before_feature_fit_no_zero_fill"}
(card/"field_semantics_correction.json").write_text(json.dumps(correction,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({k:correction[k] for k in ["zero_rate_rows","zero_rate_by_year","zero_rate_rows_with_positive_volume_and_turnover","Stage1_train_nonzero_known","Stage1_train_rows","decision"]},ensure_ascii=False,indent=2))

[74280:MainThread](2026-10-23 22:49:00,303) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[74280:MainThread](2026-10-23 22:49:00,305) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[74280:MainThread](2026-10-23 22:49:00,305) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_futu_soxl_turnover_20261023')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


[74280:MainThread](2026-10-23 22:49:00,354) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[74280:MainThread](2026-10-23 22:49:00,355) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[74280:MainThread](2026-10-23 22:49:00,355) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{
  "zero_rate_rows": 444,
  "zero_rate_by_year": {
    "2010": 192,
    "2011": 252
  },
  "zero_rate_rows_with_positive_volume_and_turnover": 444,
  "Stage1_train_nonzero_known": 2766,
  "Stage1_train_rows": 3020,
  "decision": "close_turnover_rate_field_before_feature_fit_no_zero_fill"
}


## 中文结论

SOXL `turnover_rate` 在2010–2011年连续444行报零，同时成交量和成交额均为正；2011年252个训练日不能当真实零换手。有效训练覆盖降为2766/3020，低于原计划3000日门，固定的同字段相对换手率版本在拟合前关闭。原数值来源记录和 NAS 备份保留，`field_semantics_correction.json` 优先于此前仅检查有限非负的覆盖结论。